In [2]:
import pandas as pd
import requests

from pathlib import Path
import json
import sys
sys.path.append(r"C:\Users\F.Turner\Documents\00. Analyses")
import use_funcs
from use_funcs import find_project_root

from api_helpers.uis_api import *

In [3]:
PROJECT_ROOT = find_project_root(Path.cwd())
CONFIG_PATH = PROJECT_ROOT / "path_config.json"

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    PATHS = json.load(f)

IMP_DIR = (PROJECT_ROOT / PATHS["imports_dir"]).resolve()
HO_DIR = (PROJECT_ROOT / PATHS["handoff_dir"]).resolve()
EXP_DIR = (PROJECT_ROOT / PATHS["exports_dir"]).resolve()
FIG_DIR = (PROJECT_ROOT / PATHS["figures_dir"]).resolve()
TAB_DIR = (PROJECT_ROOT / PATHS["tables_dir"]).resolve()

for folder in [IMP_DIR, HO_DIR, EXP_DIR, FIG_DIR, TAB_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("IMP_DIR:", IMP_DIR)
print("HO_DIR:", HO_DIR)
print("EXP_DIR:", EXP_DIR)
print("FIG_DIR:", FIG_DIR)
print("TAB_DIR:", TAB_DIR)

PROJECT_ROOT: C:\Users\F.Turner\Documents\00. Analyses\Education Financing
IMP_DIR: C:\Users\F.Turner\Documents\00. Analyses\Education Financing\Data
HO_DIR: C:\Users\F.Turner\Documents\00. Analyses\Education Financing\Handoff
EXP_DIR: C:\Users\F.Turner\Documents\00. Analyses\Education Financing\Results
FIG_DIR: C:\Users\F.Turner\Documents\00. Analyses\Education Financing\Results\figures
TAB_DIR: C:\Users\F.Turner\Documents\00. Analyses\Education Financing\Results\tables


# Outcome 1: Learning Adjusted Years of Schooling (LAYS)

In [4]:
# Fetch the data.
df = pd.read_csv("https://ourworldindata.org/grapher/learning-adjusted-years-of-schooling-gender-scatter.csv?v=1&csvType=full&useColumnShortNames=true", storage_options = {'User-Agent': 'Our World In Data data fetch/1.0'})

In [5]:
df['lays'] = (df['hd_hci_lays_ma']+ df['hd_hci_lays_fe'])/2

df = df.rename(columns={"entity":"country", 
                        'code':'iso3', 
                        'hd_hci_lays_ma':"lays_male",
                        'hd_hci_lays_fe':"lays_female"})

lays = df[['iso3', 'year', 'lays', 'lays_male', 'lays_female']]

In [6]:
lays.to_csv(IMP_DIR / "lays.csv", index=False)

# Outcome 2: Primary Completion Rate

In [7]:
iso3 = pd.read_csv(IMP_DIR / "iso3.csv")
iso_list = iso3["iso3"].tolist()

In [8]:
search = uis_search_indicators('Completion rate, primary education, both sexes (modelled data) (%)')
display(search)

display(search['indicator_name'].to_list())

,indicator_id,indicator_name,theme,last_data_update,last_data_update_description,total_record_count,timeline_min,timeline_max,entity_types
0,CR.MOD.1,"Completion rate, primary education, both sexes...",EDUCATION,2026-02-09,February 2026 Data Release,7390,1981,2025,[NATIONAL]


['Completion rate, primary education, both sexes (modelled data) (%) ']

{'Completion rate, primary education, adjusted wealth parity index (WPIA)' : CR.1.WPIA,
'Completion rate, primary education, both sexes (%)' : CR.1,
'Completion rate, primary education, both sexes (modelled data) (%)' : CR.MOD.1}

In [9]:
df = uis_get(indicators=['CR.1', 'CR.MOD.1', 'CR.1.WPIA'], 
             entities = iso_list, 
             start_year=2010, end_year=2026)

In [10]:
map = {'CR.1.WPIA' : 'pcr_wpia',
'CR.1' : 'pcr_pct',
'CR.MOD.1' : 'pcr_modelled_pct'}

df['indicator_name'] = df['indicator_id'].map(map)
df = df.drop(columns=['magnitude', 'qualifier'])
df = df.rename(columns={'entity_id':'iso3'})

In [11]:
pcr = df.pivot_table(index=['iso3', 'year'], columns='indicator_name', values='value').reset_index().rename_axis(columns=None)

In [12]:
pcr.to_csv(IMP_DIR / "pcr.csv", index=False)